In [ ]:
import pandas as pd
import re
from textblob import TextBlob
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk import ngrams

In [21]:
df = pd.read_csv("mh_wilds_reviews.csv")

In [22]:
# Function to clean text
def clean_text(text):
    
    # Ensure text is present
    if not isinstance(text, str):
        return ""
    
    # Lowercase all text
    text = text.lower()
    
    # Regex to remove anything that is not lowercase or a space
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Reduce extra spaces to one
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


In [23]:
df['clean_review'] = df['review'].apply(clean_text)

print(df.head())

   recommendationid                                             author  \
0         220264619  {'steamid': '76561198234040616', 'personaname'...   
1         220255762  {'steamid': '76561199206336965', 'personaname'...   
2         220254520  {'steamid': '76561198370063140', 'personaname'...   
3         220251838  {'steamid': '76561198103951446', 'personaname'...   
4         220251567  {'steamid': '76561198347295917', 'personaname'...   

  language                                             review  \
0  english                                          good game   
1  english                                    Good ♥♥♥♥ game!   
2  english  The most fun one, sure its easier, but who car...   
3  english    Can't wait for next Game "Monster Hunter Crash"   
4  english  Very fun game but still needs some more optimi...   

   timestamp_created  timestamp_updated  voted_up  votes_up  votes_funny  \
0         1773054238         1773054238      True         0            0   
1         17

In [24]:
# Create function to find text sentiment
def get_sentinment(text):
    
    if not text:
        return 0.0
    
    analysis = TextBlob(text)
    
    score = analysis.sentiment.polarity
    
    return score

In [25]:
# Run get_sentiment over df["clean_review"]
df["sentiment_score"] = df["clean_review"].apply(get_sentinment)

# Check to see if it worked
print(df[['clean_review', 'sentiment_score']].head(10))

                                        clean_review  sentiment_score
0                                          good game         0.150000
1                                          good game         0.150000
2  the most fun one sure its easier but who cares...         0.400000
3       cant wait for next game monster hunter crash        -0.200000
4  very fun game but still needs some more optimi...         0.163333
5  resorted to this after dauntless shutdown abso...         0.200000
6                                                            0.000000
7  i come from the xbox version of this game as a...         0.010714
8  great gameplay but they ruined it by forcing a...         0.106944
9                                                fun         0.300000


In [26]:
# Calculate the mean sentiment score
average_sentiment = df["sentiment_score"].mean()

print(average_sentiment)

0.10580425317287363


In [27]:
# Find scores that indicate moderate or higher positivity towards the game
positive_scores_df = df[df["sentiment_score"] > 0.1]

total_positive = len(positive_scores_df)

print(total_positive)

# Find scores that indicate moderate or higher negativity towards the game
negative_scores_df = df[df["sentiment_score"] < -0.1]

total_negative = len(negative_scores_df)

print(total_negative)

412
125


In [ ]:
# Use NLTK stopwords for NLP use later
nltk_stop_words = set(stopwords.words('english'))

print(nltk_stop_words)

{'him', 'they', 'there', 'because', 'through', "wasn't", 'where', 'shan', 'hasn', 'his', 'has', 'into', 'why', 'up', 'mustn', 'were', 'your', 'some', 'i', 'most', 'as', 'what', 'he', 'we', 'for', "should've", "you'd", 'before', 'having', "aren't", 'if', 'after', 'too', 'o', "she'd", 'am', 'of', 'won', 'about', 'further', 'yours', "you're", 'itself', 'shouldn', 'had', "hasn't", 'wouldn', 'herself', 'than', 'will', 'wasn', "they'll", 'in', 'hadn', 'all', "we'd", 'from', 'her', 'being', 'haven', 'ma', 'don', 'with', 'y', "hadn't", 'but', "i'll", 'my', 'yourselves', 'here', 'himself', "he'll", 'is', 'or', 'now', 'ourselves', 'no', 'only', 've', "it'd", 'can', 'doing', 're', 'isn', 'own', "don't", "that'll", 'their', 'which', 'theirs', 'them', "mustn't", 'this', "she'll", 'needn', 's', 'does', 'our', 'down', 'did', 'each', 'once', "won't", 'm', 'just', 'd', 'a', "i've", "shan't", 'an', 'against', "doesn't", 'how', 't', 'yourself', 'll', 'been', "he's", "he'd", 'when', "didn't", "i'd", "i'm"

In [36]:
# Add in custom stop words for MHWilds
custom_stops = {
    'game', 'games', 'play', 'playing', 'played', 
    'monster', 'hunter', 'wilds', 'capcom', 
    'im', 'like', 'just', 'get', 'even', 'really', 'would'
}

# Combine them into one
stop_words = nltk_stop_words.union(custom_stops)

In [ ]:
# Turn negative_scores_df to join all text for NLP to find all words in negative
# reviews
negative_string = " ".join(negative_scores_df["clean_review"])

all_negative_words = negative_string.split()

# Filter out stop words
filtered_negative_words = [word for word in all_negative_words
                           if word not in stop_words]
word_pairs = list(ngrams(filtered_negative_words, 2))

negative_counter = Counter(word_pairs)

print(negative_counter.most_common(10))

[('fun', 15), ('launch', 14), ('still', 14), ('time', 14), ('performance', 14), ('cant', 13), ('bad', 13), ('crashes', 12), ('make', 12), ('dont', 11)]
